In [2]:
import duckdb
import pandas as pd
import seaborn as sns

In [4]:
thi = pd.read_csv("tarbijahinna_indeks_2005_2025_raw.csv", sep=",")
thi

,,,,,,,,,,,,,"IA002: TARBIJAHINNAINDEKS, 1997 = 100 | Aasta, Kaubagrupp ning Kuu"
Aasta,Kaubagrupp,Jaanuar,Veebruar,Märts,Aprill,Mai,Juuni,Juuli,August,September,Oktoober,November,Detsember
1998,Kokku,105.58,106.59,107.43,107.90,108.28,108.48,109.10,108.95,108.89,109.07,109.05,109.17
1999,Kokku,110.44,110.74,111.25,111.53,111.90,111.88,111.93,111.74,111.94,112.15,112.51,113.41
2000,Kokku,114.01,114.05,114.84,114.96,115.15,115.47,116.53,116.69,117.18,118.25,118.95,119.14
2001,Kokku,120.63,121.00,121.49,122.29,123.13,123.43,123.86,123.80,123.89,123.94,123.88,124.14
2002,Kokku,125.70,126.37,126.73,127.89,128.23,128.11,127.70,127.07,127.21,127.89,127.69,127.47
2003,Kokku,128.94,129.37,129.75,129.49,129.06,128.53,128.73,128.68,128.97,129.18,128.99,128.83
2004,Kokku,129.55,129.95,130.22,131.16,133.87,134.33,133.96,133.85,134.03,134.49,135.04,135.27
2005,Kokku,135.03,135.82,136.73,137.30,137.56,138.53,139.04,139.45,140.59,140.53,140.32,140.08
2006,Kokku,141.31,142.01,142.15,143.21,143.98,144.51,145.30,146.41,145.87,145.69,146.84,147.28


In [12]:
try:
    thi = pd.read_csv("tarbijahinna_indeks_2005_2025_raw.csv", sep=",")
    if "IA002" in str(thi.columns):
        thi = pd.read_csv("tarbijahinna_indeks_2005_2025_raw.csv", sep=",", skiprows=1)
except Exception:
    thi = pd.read_csv("tarbijahinna_indeks_2005_2025_raw.csv", sep=",")

# Kirjutame päise kohe puhtalt üle, et vältida nimevigu
kuu_nimed = [str(col).replace('"', '').strip() for col in thi.columns[2:]]
thi.columns = ["Aasta", "Kaubagrupp"] + kuu_nimed

# Eemaldame jutumärgid aastaarvudest
thi["Aasta"] = thi["Aasta"].astype(str).str.replace('"', '').str.strip()

# 2. Teisendame laia tabeli pikaks formaadiks (Aastad ja kuud ridadena)
thi_long = thi.melt(
    id_vars=["Aasta"], # Võtame identifikaatoriks ainult Aasta (Kaubagrupp jääb välja)
    value_vars=kuu_nimed,
    var_name="Kuu", 
    value_name="Indeks"
)

# 3. Puhastame indeksite väärtused numbriteks (asendame ".." tühjusega)
thi_long["Indeks"] = pd.to_numeric(thi_long["Indeks"].astype(str).str.replace('"', ''), errors='coerce')

# 4. Seame kuud õigesse kronoloogilisse järjekorda
kuude_jarjekord = ["Jaanuar", "Veebruar", "Märts", "Aprill", "Mai", "Juuni", "Juuli", "August", "September", "Oktoober", "November", "Detsember"]
thi_long["Kuu"] = pd.Categorical(thi_long["Kuu"], categories=kuude_jarjekord, ordered=True)

# 5. Sorteerime andmed nii, et aastad ja kuud jookseksid loogilises ajalises jadas
thi_flat = thi_long.sort_values(by=["Aasta", "Kuu"]).reset_index(drop=True)

# Kuvame tulemuse
thi_flat

,Aasta,Kuu,Indeks
0,1998,Jaanuar,105.58
1,1998,Veebruar,106.59
2,1998,Märts,107.43
3,1998,Aprill,107.90
4,1998,Mai,108.28
...,...,...,...
343,2026,August,326.25
344,2026,September,NaN
345,2026,Oktoober,NaN
346,2026,November,NaN


In [16]:
import duckdb
import pandas as pd
import seaborn as sns

# 1. Loeme andmed sisse ja puhastame veerunimed
try:
    thi = pd.read_csv("tarbijahinna_indeks_2005_2025_raw.csv", sep=",")
    if "IA002" in str(thi.columns):
        thi = pd.read_csv("tarbijahinna_indeks_2005_2025_raw.csv", sep=",", skiprows=1)
except Exception:
    thi = pd.read_csv("tarbijahinna_indeks_2005_2025_raw.csv", sep=",")

# Kirjutame päise puhtalt üle
kuu_nimed = [str(col).replace('"', '').strip() for col in thi.columns[2:]]
thi.columns = ["Aasta", "Kaubagrupp"] + kuu_nimed
thi["Aasta"] = thi["Aasta"].astype(str).str.replace('"', '').str.strip()

# 2. Teisendame laia tabeli pikaks formaadiks (ilma Kaubagrupita)
thi_long = thi.melt(
    id_vars=["Aasta"], 
    value_vars=kuu_nimed,
    var_name="Kuu", 
    value_name="Indeks"
)

# 3. Puhastame indeksite väärtused numbriteks
thi_long["Indeks"] = pd.to_numeric(thi_long["Indeks"].astype(str).str.replace('"', ''), errors='coerce')

# 4. Sorteerime kuud kronoloogiliselt
kuude_jarjekord = ["Jaanuar", "Veebruar", "Märts", "Aprill", "Mai", "Juuni", "Juuli", "August", "September", "Oktoober", "November", "Detsember"]
thi_long["Kuu"] = pd.Categorical(thi_long["Kuu"], categories=kuude_jarjekord, ordered=True)
thi_flat = thi_long.sort_values(by=["Aasta", "Kuu"]).reset_index(drop=True)

# 5. MÄÄRAME KVARTALI ALGUSKUU PÄEVA ( quarter_id )
# Seome iga kuu otseselt tema kvartali esimese päeva kuuga (MM-DD)
kvartali_alguskuud = {
    "Jaanuar": "01-01", "Veebruar": "01-01", "Märts": "01-01",
    "Aprill": "04-01", "Mai": "04-01", "Juuni": "04-01",
    "Juuli": "07-01", "August": "07-01", "September": "07-01",
    "Oktoober": "10-01", "November": "10-01", "Detsember": "10-01"
}

# Paneme kokku quarter_id veeru: Aasta + kvartali alguse kuu ja päev
thi_flat["quarter_id"] = thi_flat["Aasta"] + "-" + thi_flat["Kuu"].map(kvartali_alguskuud)

# Kuvame puhta ja sorteeritud tulemuse
thi_flat


,Aasta,Kuu,Indeks,quarter_id
0,1998,Jaanuar,105.58,1998-01-01
1,1998,Veebruar,106.59,1998-01-01
2,1998,Märts,107.43,1998-01-01
3,1998,Aprill,107.90,1998-04-01
4,1998,Mai,108.28,1998-04-01
...,...,...,...,...
343,2026,August,326.25,2026-07-01
344,2026,September,NaN,2026-07-01
345,2026,Oktoober,NaN,2026-10-01
346,2026,November,NaN,2026-10-01


In [21]:
# Kirjutame SQL päringu DuckDB abil
# Filtreerime aastad 2005-2025, grupeerime 'quarter_id' järgi ja arvutame keskmise indeksi
sql_query = """
    SELECT 
        quarter_id,
        ROUND(AVG(Indeks), 2) AS keskmine_indeks
    FROM thi_flat
    WHERE Indeks IS NOT NULL
      AND quarter_id BETWEEN '2005-01-01' AND '2025-12-31'
    GROUP BY quarter_id
    ORDER BY quarter_id ASC;
"""

# Käivitame päringu ja salvestame tulemuse uude Pandase andmeraami
kvartalite_keskmised = duckdb.query(sql_query).to_df()

# Kuvame tulemuse
kvartalite_keskmised

# Salvestame tulemuse CSV-failina
kvartalite_keskmised.to_csv("kvartalite_keskmised_2005_2025.csv", index=False)

print("Fail 'kvartalite_keskmised_2005_2025.csv' on edukalt salvestatud!")

Fail 'kvartalite_keskmised_2005_2025.csv' on edukalt salvestatud!
